In [17]:
from training_utilities_2nd_part import *

In [18]:
# Zurich
from variables_to_specify_zurich import *
df, columns_to_normalize, zurich_target_col, forecast_avg_target_col_name, avg_target_col_name, No_of_datapoints_in_one_day, start_date, end_date, delta, one_month_days, out_columns, zurich_drop_columnss, zurich_windows, index_of_one_month, one_month_window_size, date_col_name = variables_to_specify_zurich()
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

df[columns_to_normalize] = scaler.fit_transform(df[columns_to_normalize])

df = df.dropna().reset_index(drop=True)
zurich_df = df
zurich_time_steps = 1
zurich_df['Timestamp'] = pd.to_datetime(zurich_df['Timestamp'])
convert_time(zurich_df, date_col_name)
zurich_df= zurich_df[zurich_df['Timestamp'].dt.year == 2020]
zurich_df = preprocess_feature_names(zurich_df)

Value_NE5


# stationary

In [20]:
zurich_len_of_training_data_of_stationary_model =14*No_of_datapoints_in_one_day

train = zurich_df[0:zurich_len_of_training_data_of_stationary_model] 
test = zurich_df[zurich_len_of_training_data_of_stationary_model:]

eval_df_first_month, stationary_model1 = stationary_model_with_hptuning(train, test, one_month_window_size, 2, out_columns, zurich_target_col, zurich_drop_columnss)


Model Type: XGBRegressor
Storage Required: 0.28 MB
model storage is : 0.2830390930175781


total_time is:  0.23501533299997845


# Model reuse

In [21]:
# Model reuse

daily_df_avg = get_elect_daily_avg(zurich_df, No_of_datapoints_in_one_day, zurich_target_col, avg_target_col_name)

seasonality_periods_acf_ls, seasonality_periods_acf, segmented_daily_df_avg, filtered_most_similar_dict_wass, filtered_most_similar_dict_tvd, forecast_daily_df_avg, segmented_forecast_daily_df_avg, filtered_forecasted_most_similar_dict_wass, filtered_forecasted_most_similar_dict_tvd = get_seasonality_segments_and_similarities(daily_df_avg, avg_target_col_name, forecast_avg_target_col_name, 14)

Detected seasonality periods (ACF): [  7  14  21  28  35  42  49  56  63  70  77  84  91  98 105 112 119 126
 133 140 147 154 161 168 175 182]
median_value is:  94


## drift detection

In [22]:
df_copy = zurich_df[[zurich_target_col]]
target_col = zurich_target_col
time_steps = zurich_time_steps

df_copy['date'] = pd.to_datetime(df_copy.index)
window_length=[14]
multiplier = No_of_datapoints_in_one_day

x = 14* multiplier
window_len_=[x]

window_length = [x]

drift_results_df_ls = []
for i in window_len_:
    start_drift_detection_time = timeit.default_timer()
    drift_results_df = detect_drift_univariate(
        df_copy,
        target_col=zurich_target_col,
        window_lengths=window_len_,
        arima_order=(1, 0, 0)
    )
    drift_results_df_ls.append(drift_results_df)
    drift_detection_time = timeit.default_timer() - start_drift_detection_time
    num_true = drift_results_df['drift_detected'].sum()
    print("i is: ", i, " and the Number of True values in 'drift_detected':", num_true, " total number of rows are : ", len(drift_results_df))
    print("drift detection time is: ", drift_detection_time)
    drift_results_df = drift_results_df_ls[0]
    drift_indices = list(drift_results_df.index[drift_results_df['drift_detected']])
    print("indices are: ", drift_indices)

Fold 0: Train size=268, Test size=268
Fold 1: Train size=536, Test size=268
Fold 2: Train size=804, Test size=268
Fold 3: Train size=1072, Test size=268
Skipping fold 4: Insufficient training or test data.
Fold 0: Train size=268, Test size=268
Fold 1: Train size=536, Test size=268
Fold 2: Train size=804, Test size=268
Fold 3: Train size=1072, Test size=268
Skipping fold 4: Insufficient training or test data.
Fold 0: Train size=268, Test size=268
Fold 1: Train size=536, Test size=268
Fold 2: Train size=804, Test size=268
Fold 3: Train size=1072, Test size=268
Skipping fold 4: Insufficient training or test data.
Fold 0: Train size=268, Test size=268
Fold 1: Train size=536, Test size=268
Fold 2: Train size=804, Test size=268
Fold 3: Train size=1072, Test size=268
Skipping fold 4: Insufficient training or test data.
Fold 0: Train size=268, Test size=268
Fold 1: Train size=536, Test size=268
Fold 2: Train size=804, Test size=268
Fold 3: Train size=1072, Test size=268
Skipping fold 4: Insuff

In [23]:
eval_df_monthly2, avg_ml_storage1 = new_copied_reuse_with_hptuning_no_while_loop_with_drift(filtered_most_similar_dict_wass, stationary_model1, zurich_len_of_training_data_of_stationary_model,zurich_df, "SA", zurich_target_col, zurich_drop_columnss, zurich_time_steps, seasonality_periods_acf, No_of_datapoints_in_one_day, drift_indices, 2)

window is:  1344
i/window is :  1.0
Model Type: XGBRegressor
Storage Required: 0.28 MB


window is:  2688
i/window is :  2.0
Model Type: XGBRegressor
Storage Required: 0.28 MB


window is:  4032
i/window is :  3.0
Model Type: XGBRegressor
Storage Required: 0.27 MB


window is:  5376
i/window is :  4.0
similar_month_index is :  2
month_index:  4



window is:  6720
i/window is :  5.0
similar_month_index is :  0
month_index:  5




window is:  8064
i/window is :  6.0
Model Type: XGBRegressor
Storage Required: 0.28 MB


window is:  9408
i/window is :  7.0
Model Type: XGBRegressor
Storage Required: 0.27 MB


window is:  10752
i/window is :  8.0
similar_month_index is :  6
month_index:  8




window is:  12096
i/window is :  9.0
similar_month_index is :  5
previous_model_i is :  12096
math.floor(previous_model_i/window) is:  9
len(models_ls) is: 8
Model Type: XGBRegressor
Storage Required: 0.28 MB


window is:  13440
i/window is :  10.0
Model Type: XGBRegressor
Storage Required: 0.28 MB


w

In [24]:
eval_df_monthly2, avg_ml_storage2 = new_copied_reuse_with_hptuning_no_while_loop_with_drift(filtered_most_similar_dict_tvd, stationary_model1, zurich_len_of_training_data_of_stationary_model,zurich_df, "SA", zurich_target_col, zurich_drop_columnss, zurich_time_steps, seasonality_periods_acf, No_of_datapoints_in_one_day, drift_indices, 2)

window is:  1344
i/window is :  1.0
Model Type: XGBRegressor
Storage Required: 0.28 MB


window is:  2688
i/window is :  2.0
Model Type: XGBRegressor
Storage Required: 0.28 MB


window is:  4032
i/window is :  3.0
Model Type: XGBRegressor
Storage Required: 0.27 MB


window is:  5376
i/window is :  4.0
Model Type: XGBRegressor
Storage Required: 0.30 MB


window is:  6720
i/window is :  5.0
similar_month_index is :  3
month_index:  5




window is:  8064
i/window is :  6.0
similar_month_index is :  3
month_index:  6




window is:  9408
i/window is :  7.0
similar_month_index is :  2
month_index:  7




window is:  10752
i/window is :  8.0
similar_month_index is :  3
month_index:  8




window is:  12096
i/window is :  9.0
similar_month_index is :  1
month_index:  9



window is:  13440
i/window is :  10.0
similar_month_index is :  2
month_index:  10




window is:  14784
i/window is :  11.0
similar_month_index is :  2
month_index:  11




window is:  16128
i/window is :  12.0
Model Type:

In [25]:
eval_df_monthly2, avg_ml_storage3 = new_copied_reuse_with_hptuning_no_while_loop_with_drift(filtered_forecasted_most_similar_dict_wass, stationary_model1, zurich_len_of_training_data_of_stationary_model,zurich_df, "ES", zurich_target_col, zurich_drop_columnss, zurich_time_steps, seasonality_periods_acf, No_of_datapoints_in_one_day, drift_indices, 2)

window is:  1344
Model Type: XGBRegressor
Storage Required: 0.28 MB


window is:  2688
Model Type: XGBRegressor
Storage Required: 0.28 MB


window is:  4032
Model Type: XGBRegressor
Storage Required: 0.27 MB


window is:  5376
similar_month_index is :  1
month_index:  3




window is:  6720
similar_month_index is :  2
month_index:  4



window is:  8064
Model Type: XGBRegressor
Storage Required: 0.28 MB


window is:  9408
Model Type: XGBRegressor
Storage Required: 0.27 MB


window is:  10752
Model Type: XGBRegressor
Storage Required: 0.27 MB


window is:  12096
Model Type: XGBRegressor
Storage Required: 0.29 MB


window is:  13440
Model Type: XGBRegressor
Storage Required: 0.28 MB


window is:  14784
similar_month_index is :  7
month_index:  10




window is:  16128
similar_month_index is :  7
month_index:  11




window is:  17472
similar_month_index is :  10
previous_model_i is :  17472
math.floor(previous_model_i/window) is:  13
len(models_ls) is: 12
Model Type: XGBRegressor
Storage

In [26]:
eval_df_monthly2, avg_ml_storage4 = new_copied_reuse_with_hptuning_no_while_loop_with_drift(filtered_forecasted_most_similar_dict_tvd, stationary_model1, zurich_len_of_training_data_of_stationary_model,zurich_df, "ES", zurich_target_col, zurich_drop_columnss, zurich_time_steps, seasonality_periods_acf, No_of_datapoints_in_one_day, drift_indices, 2)

window is:  1344
Model Type: XGBRegressor
Storage Required: 0.28 MB


window is:  2688
Model Type: XGBRegressor
Storage Required: 0.28 MB


window is:  4032
Model Type: XGBRegressor
Storage Required: 0.27 MB


window is:  5376
Model Type: XGBRegressor
Storage Required: 0.30 MB


window is:  6720
Model Type: XGBRegressor
Storage Required: 0.29 MB


window is:  8064
Model Type: XGBRegressor
Storage Required: 0.28 MB


window is:  9408
Model Type: XGBRegressor
Storage Required: 0.27 MB


window is:  10752
Model Type: XGBRegressor
Storage Required: 0.27 MB


window is:  12096
similar_month_index is :  6
month_index:  8




window is:  13440
similar_month_index is :  2
month_index:  9



window is:  14784
Model Type: XGBRegressor
Storage Required: 0.27 MB


window is:  16128
similar_month_index is :  8
previous_model_i is :  16128
math.floor(previous_model_i/window) is:  12
len(models_ls) is: 11
Model Type: XGBRegressor
Storage Required: 0.27 MB


window is:  17472
similar_month_index is : 

In [27]:
avg_ml_storage_reuse = (avg_ml_storage1+avg_ml_storage2+avg_ml_storage3+avg_ml_storage4)/4
print(avg_ml_storage_reuse)

0.27962679680144964


# informed

In [28]:
informed_update(stationary_model1,zurich_df, target_col, zurich_drop_columnss,time_steps, seasonality_periods_acf,No_of_datapoints_in_one_day, drift_indices, 2)

window is:  1344
Model Type: XGBRegressor
Storage Required: 0.28 MB
window is:  2688
Model Type: XGBRegressor
Storage Required: 0.28 MB
window is:  4032
Model Type: XGBRegressor
Storage Required: 0.27 MB
window is:  5376
window is:  6720
window is:  8064
window is:  9408
window is:  10752
Model Type: XGBRegressor
Storage Required: 0.27 MB
window is:  12096
window is:  13440
Model Type: XGBRegressor
Storage Required: 0.28 MB
window is:  14784
Model Type: XGBRegressor
Storage Required: 0.27 MB
window is:  16128
Model Type: XGBRegressor
Storage Required: 0.28 MB
window is:  17472
Model Type: XGBRegressor
Storage Required: 0.27 MB
window is:  18816
Model Type: XGBRegressor
Storage Required: 0.29 MB
window is:  20160
Model Type: XGBRegressor
Storage Required: 0.28 MB
window is:  21504
window is:  22848
Model Type: XGBRegressor
Storage Required: 0.27 MB
window is:  24192
Model Type: XGBRegressor
Storage Required: 0.29 MB
window is:  25536
Model Type: XGBRegressor
Storage Required: 0.27 MB
wi

# periodical

In [29]:
periodical_retraining_with_hptuning(2, zurich_df, zurich_windows, out_columns, zurich_target_col, zurich_drop_columnss)

window is : 480
window size is :  480
Model Type: XGBRegressor
Storage Required: 0.12 MB
Model Type: XGBRegressor
Storage Required: 0.14 MB
Model Type: XGBRegressor
Storage Required: 0.13 MB
Model Type: XGBRegressor
Storage Required: 0.12 MB
Model Type: XGBRegressor
Storage Required: 0.14 MB
Model Type: XGBRegressor
Storage Required: 0.13 MB
Model Type: XGBRegressor
Storage Required: 0.13 MB
Model Type: XGBRegressor
Storage Required: 0.12 MB
Model Type: XGBRegressor
Storage Required: 0.13 MB
Model Type: XGBRegressor
Storage Required: 0.12 MB
Model Type: XGBRegressor
Storage Required: 0.13 MB
Model Type: XGBRegressor
Storage Required: 0.13 MB
Model Type: XGBRegressor
Storage Required: 0.13 MB
Model Type: XGBRegressor
Storage Required: 0.12 MB
Model Type: XGBRegressor
Storage Required: 0.12 MB
Model Type: XGBRegressor
Storage Required: 0.12 MB
Model Type: XGBRegressor
Storage Required: 0.11 MB
Model Type: XGBRegressor
Storage Required: 0.12 MB
Model Type: XGBRegressor
Storage Required: 0

([         Training dataset     Testing dataset       mae       mse      rmse  \
  0   trained on window i-1  tested on window i  0.183281  0.054389  0.233213   
  1   trained on window i-1  tested on window i  0.080718  0.011599  0.107697   
  2   trained on window i-1  tested on window i  0.073258  0.009143  0.095619   
  3   trained on window i-1  tested on window i  0.069343  0.008056  0.089757   
  4   trained on window i-1  tested on window i  0.098978  0.014728  0.121357   
  ..                    ...                 ...       ...       ...       ...   
  67  trained on window i-1  tested on window i  0.078869  0.009103  0.095411   
  68  trained on window i-1  tested on window i  0.073858  0.010430  0.102127   
  69  trained on window i-1  tested on window i  0.092697  0.014304  0.119598   
  70  trained on window i-1  tested on window i  0.110243  0.020484  0.143123   
  71  trained on window i-1  tested on window i  0.085314  0.011929  0.109220   
  
            r2       mape